# 18 · Build the combined knockout object

Concatenates the single- and multiple-knockout populations, restricts to the
module gene set, residualises out the cluster and QC covariates, and labels
each cell with the guide module it was perturbed in.

**Reads** `par_save_filename_8` and `par_em_selected_cells_file`, subsetting as
notebook 15 does, together with `par_save_filename_9`.
**Writes** `par_save_filename_11` and `par_save_filename_12`.

The `K_0` … `K_5` columns are the guide-module numbering from notebook 17. In
the published analysis they map to the manuscript's module names as K0=M2,
K1=M3, K2=M6, K3=M5, K4=M1, K5=M4. Relabel at the point of drawing a figure,
not here.

## Setup

In [ ]:
from libraries import *
from parameters import *
from pathlib import Path

os.chdir(projectDir)
from sklearn import linear_model
import anndata as ad

## Concatenate the two populations

An inner join on genes. The single-knockout object's guide and gene column
lists carry over, since the multiple-knockout object was built from them in
notebook 13.

In [ ]:
# The per-gene object from notebook 13, subset to the EM-selected cells, as in
# notebook 15. The two must be fitted on the same population.
adata_single = sc.read(par_save_filename_8)
selected = pd.read_csv(par_em_selected_cells_file).iloc[:, 0].astype(str)
adata_single = adata_single[adata_single.obs_names.isin(set(selected))].copy()
print(f"single  : {adata_single.shape} (EM-selected)")

adata_multiple = sc.read(par_save_filename_9)
print(f"multiple: {adata_multiple.shape}")

adata_single.obs["GENE_INEFFECT_"] = 0

adata_all = ad.concat([adata_single, adata_multiple], join="inner",
                      label="batch", keys=["0", "1"], index_unique="-")
adata_all.uns["feature_barcode_names_filtered"] = adata_single.uns["feature_barcode_names_filtered"]
adata_all.uns["feature_barcode_names_filtered_GENES"] = adata_single.uns["feature_barcode_names_filtered_GENES"]
print(f"combined: {adata_all.shape}")

## Label the DC subtypes

In [ ]:
for subtype, clusters in par_subcelltypes.items():
    adata_all.obs[subtype] = adata_all.obs.leiden.isin(clusters).astype("int64")

print(adata_all.obs[list(par_subcelltypes)].sum().to_string())

adata_all.write(par_save_filename_11)
print(f"written: {par_save_filename_11}")

## Restrict to the module gene set

The response genes are those assigned to a gene module in notebook 17, plus the
genes listed in `par_extra_response_genes`.

In [ ]:
module_genes = pd.read_csv(par_geneModules_file, header=0, index_col=0).iloc[:, 0]
wanted = pd.concat([module_genes, pd.Series(par_extra_response_genes)])
keep = [g for g in wanted if g in adata_all.var_names]

adata_sub = adata_all[:, keep].copy()
print(f"reduced to {adata_sub.shape[1]} response genes")

## Residualise out cluster and QC covariates

The `ClusterResiduals` layer is recomputed here because the gene set changed.
Every combinatorial model is fitted on this layer rather than on `.X`.

In [ ]:
rna = pd.DataFrame(
    adata_sub.X.toarray() if hasattr(adata_sub.X, "toarray") else adata_sub.X,
    index=adata_sub.obs_names, columns=adata_sub.var_names,
)
covars = adata_sub.obs[["n_genes", "mt_frac", "leiden"]]
covars = covars.join(pd.get_dummies(covars.leiden)).drop(columns=["leiden"])

regr = linear_model.LinearRegression(fit_intercept=False).fit(covars, rna)
residuals = rna - regr.predict(covars)
adata_sub.layers["ClusterResiduals"] = residuals.to_numpy()

print(f"ClusterResiduals recomputed: {adata_sub.layers['ClusterResiduals'].shape}")
print(f"mean residual: {residuals.to_numpy().mean():.3e} (should be ~0)")

## Assign each cell its guide module

Only cells whose knockouts all fall inside the module table are kept: a cell
carrying one module guide and one guide from outside is not interpretable as a
module combination.

Cells are then classified three ways — one perturbed module, two guides in the
same module, two guides in different modules — flagged by `Doubles` and
`DoubleSameGroup`.

In [ ]:
guide_modules = pd.read_csv(par_guideModules_file, index_col=0)
guide_modules.GuideName = "GENE_" + guide_modules.GuideName + "_"
guide_modules.index = guide_modules.GuideName

gene_cols = adata_sub.uns["feature_barcode_names_filtered_GENES"]
indicators = adata_sub.obs[gene_cols]

in_module = [c for c in guide_modules.GuideName.tolist() + ["GENE_CONTROL_"]
             if c in indicators.columns]
outside = [c for c in indicators.columns if c not in in_module]

keep = (indicators[in_module].sum(axis=1) > 0) & (indicators[outside].sum(axis=1) == 0)
indicators = indicators.loc[keep, in_module].copy()
print(f"cells carrying only module guides: {indicators.shape[0]}")

n_targets = indicators.sum(axis=1)
singles = indicators.loc[n_targets == 1].T
doubles = indicators.loc[n_targets == 2].T
print(f"  singles: {singles.shape[1]}, doubles: {doubles.shape[1]}")

In [ ]:
group_of = guide_modules.GuideGroup

def by_module(frame, threshold):
    """Collapse gene rows onto their module; controls pass through."""
    perturbed = frame.loc[~frame.index.isin(["GENE_CONTROL_"])]
    grouped = perturbed.groupby(group_of).sum() > threshold
    return pd.concat([grouped, frame.loc[["GENE_CONTROL_"]]], axis=0).T

grouped_singles = by_module(singles, 0)
grouped_singles["Doubles"] = 0
grouped_singles["DoubleSameGroup"] = 0

same_group = by_module(doubles, 1)
same_group = same_group.loc[same_group.sum(axis=1) == 1]
same_group["Doubles"] = 1
same_group["DoubleSameGroup"] = 1

diff_group = by_module(doubles, 0)
diff_group = diff_group.loc[diff_group.sum(axis=1) == 2]
diff_group["Doubles"] = 1
diff_group["DoubleSameGroup"] = 0

final = pd.concat([grouped_singles, diff_group, same_group])
module_names = [f"K_{i}" for i in range(final.shape[1] - 3)]
final.columns = module_names + ["K_CONTROL", "Doubles", "DoubleSameGroup"]
final = final.astype("int64")

print(f"single-module cells  : {grouped_singles.shape[0]}")
print(f"same-module doubles  : {same_group.shape[0]}")
print(f"cross-module doubles : {diff_group.shape[0]}")
print(f"\ncells per module:\n{final[module_names + ['K_CONTROL']].sum().to_string()}")

## Write the model input object

In [ ]:
adata_model = adata_sub[final.index, :].copy()
adata_model.obs = adata_model.obs.join(final)

adata_model.write(par_save_filename_12)
print(f"written: {par_save_filename_12}  ({adata_model.shape[0]} x {adata_model.shape[1]})")
print(f"layers: {list(adata_model.layers)}")